In [1]:
"""
Inspect the WiCount dataset (ESP32-S3 CSI, people count 0-3).

Goal: check the things the dataset card does NOT tell us:
  - how many CSV files per class (this decides how we can split train/test)
  - real packet rate
  - which MAC addresses appear (is the transmitter a router or an ESP32?)
  - does each CSI row really have 104 numbers (52 complex subcarriers)?

"""
import json
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

# 1. Download the dataset (cached after the first run)
zip_path = hf_hub_download(
    repo_id="RS2002/WiCount",
    repo_type="dataset",         
    filename="WiCount.zip",
    local_dir="../data/raw/WiCount",
)

out = Path("../data/raw/WiCount/extracted")
with zipfile.ZipFile(zip_path) as z:
    z.extractall(out)



def parse_csi(cell):
    """The 'data' column holds a list of numbers stored as text."""
    if isinstance(cell, str):
        try:
            return json.loads(cell)
        except json.JSONDecodeError:
            return [int(x) for x in cell.strip("[] ").split(",") if x.strip()]
    return list(cell)


# 2. Look at every CSV file
records = []
macs_per_class = {}
for csv_path in sorted(out.rglob("*.csv")):
    label = csv_path.parent.name          # folder name = number of people
    df = pd.read_csv(csv_path)

    # duration from the 'timestamp' column (UTC+8 time of collection)
    ts = pd.to_datetime(df["timestamp"], errors="coerce").dropna()
    seconds = (ts.iloc[-1] - ts.iloc[0]).total_seconds() if len(ts) > 1 else np.nan

    # length of the CSI vector in the first row
    csi_len = len(parse_csi(df["data"].iloc[0]))

    # MAC address column (name may differ, so search for it)
    mac_cols = [c for c in df.columns if "mac" in c.lower()]
    if mac_cols:
        macs_per_class.setdefault(label, set()).update(df[mac_cols[0]].unique())

    records.append({
        "class": label,
        "file": csv_path.name,
        "rows": len(df),
        "seconds": round(seconds, 1),
        "rate_hz": round(len(df) / seconds, 1) if seconds and seconds > 0 else np.nan,
        "csi_len": csi_len,
        "rssi_mean": round(df["rssi"].mean(), 1),
    })

info = pd.DataFrame(records)

# 3. Print the summary
print("\n=== Per file ===")
print(info.to_string(index=False))

print("\n=== Per class ===")
summary = info.groupby("class").agg(
    files=("file", "count"),
    total_rows=("rows", "sum"),
    total_minutes=("seconds", lambda s: round(s.sum() / 60, 1)),
    mean_rate_hz=("rate_hz", "mean"),
    rssi_mean=("rssi_mean", "mean"),
)
print(summary.to_string())

print("\n=== CSI vector lengths seen ===", sorted(info["csi_len"].unique()))
print("\n=== MAC addresses per class ===")
for label, macs in sorted(macs_per_class.items()):
    print(label, sorted(macs))

WiCount.zip: reconstructing file:   0%|          |  0.00B / 9.39MB            

WiCount.zip: downloading bytes:           |  0.00B            


=== Per file ===
class                              file  rows  seconds  rate_hz  csi_len  rssi_mean
    0 2023-12-21_19-23-53-985_104_1.csv  5218     60.0     87.0      104      -27.2
    1 2023-12-21_19-27-21-930_104_1.csv  4780     60.0     79.7      104      -28.0
    1 2023-12-21_19-31-17-706_104_1.csv  5189     60.0     86.5      104      -27.9
    1 2023-12-21_19-34-42-783_104_1.csv  5234     60.0     87.2      104      -27.8
    1 2023-12-21_19-37-36-861_104_1.csv  5163     60.0     86.1      104      -27.7
    1 2023-12-21_19-40-12-110_104_1.csv  5421     60.0     90.4      104      -27.6
    1 2023-12-21_19-42-28-267_104_1.csv  5644     60.0     94.1      104      -27.6
    1 2023-12-21_19-45-07-501_104_1.csv  5128     60.0     85.5      104      -27.9
    2 2023-12-21_19-47-14-669_104_1.csv  5376     60.0     89.6      104      -29.1
    2 2023-12-21_19-49-37-940_104_1.csv  5307     60.0     88.5      104      -28.4
    2 2023-12-21_19-54-01-269_104_1.csv  5141     60.0    